# Assignment 1.2: AI Agents and AI Literature
**Francisco Gutierrez**

Question 1 documents the literature collection workflow. Question 2(a) contains the agent implementation; Questions 2(b) and 2(c) define its proposed goal and future functionality. Agent execution is not required. An optional execution attempt returned a Gemini 503 availability error; no successful agent actions are claimed.


# Question 2: AI Agent

## Question 2(a): LlamaIndex and Gemini implementation

The agent observes assignment deadlines, calendar availability, and course notes, then creates a task, reserves a study block, and saves a study plan. Gemini selects tools through LlamaIndex's `FunctionAgent`.

| Type | Tool | Purpose |
|---|---|---|
| Sensor | `read_assignments` | Read simulated deadlines and work estimates |
| Sensor | `read_calendar` | Read simulated study availability |
| Sensor | `read_course_notes` | Read simulated assignment requirements |
| Actuator | `create_task` | Write a local task record |
| Actuator | `reserve_study_block` | Write a local study reservation |
| Actuator | `save_study_plan` | Save a Markdown plan |

Inputs are simulated. Actuators implement real local file writes, without connecting to an external calendar or MCP server. An optional execution attempt reached Gemini but returned a 503 high-demand error. No successful agent actions were observed. Execution is not required by the assignment. No API key is included in this notebook.


In [2]:
# Optional setup; execution is not required.
# %pip install llama-index-core llama-index-llms-google-genai python-dotenv


  Using cached defusedxml-0.7.1-py2.py3-none-any.whl.metadata (32 kB)
  Using cached click-8.5.0-py3-none-any.whl.metadata (2.6 kB)
  Using cached greenlet-3.5.6-cp312-cp312-win_amd64.whl.metadata (3.9 kB)
   ---------------------------------------- 0.0/11.9 MB ? eta -:--:--
   -- ------------------------------------- 0.8/11.9 MB 8.3 MB/s eta 0:00:02
   ----------- ---------------------------- 3.4/11.9 MB 11.8 MB/s eta 0:00:01
   ----------------------- ---------------- 7.1/11.9 MB 14.1 MB/s eta 0:00:01
   -------------------------------- ------- 9.7/11.9 MB 13.7 MB/s eta 0:00:01
   ---------------------------------------- 11.9/11.9 MB 13.6 MB/s  0:00:00
   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   ---------------------------------------- 1.7/1.7 MB 23.6 MB/s  0:00:00
Using cached greenlet-3.5.6-cp312-cp312-win_amd64.whl (324 kB)
   ---------------------------------------- 0.0/941.2 kB ? eta -:--:--
   ---------------------------------------- 941.2/941.2 kB 1

In [4]:
"""Study-planner agent example. No API call occurs unless main() is run.

Install if you want to run it later:
python -m pip install llama-index-core llama-index-llms-google-genai python-dotenv

Sensors: assignment deadlines, calendar availability, course notes.
Actuators: create a task, reserve a study block, save a study plan.
The small input dataset is simulated; the output actions write real local files.
"""

import asyncio
import json
import os
from datetime import datetime
from pathlib import Path

from dotenv import load_dotenv
from llama_index.core.agent.workflow import FunctionAgent
from llama_index.core.tools import FunctionTool
from llama_index.llms.google_genai import GoogleGenAI


# Locate the USD root from the notebook working directory.
USD_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents) if (path / ".env").is_file()),
    None,
)
if USD_ROOT is None:
    raise FileNotFoundError("Start the notebook within USD, with its root .env present.")
OUTPUT_DIR = USD_ROOT / "AAI_501" / "M1" / "study_agent_output"


# SENSORS: observe three different kinds of information.
def read_assignments() -> str:
    """Read the simulated assignment tracker, including deadlines and work estimates."""
    return json.dumps([
        {"course": "AAI 501", "assignment": "AI agent", "due": "2026-11-02",
         "estimated_hours": 3, "status": "not started"},
        {"course": "AAI 500", "assignment": "Statistics review", "due": "2026-11-04",
         "estimated_hours": 2, "status": "not started"},
    ])


def read_calendar() -> str:
    """Read simulated free study windows; returned times are America/Los_Angeles."""
    return json.dumps([
        {"start": "2026-10-30T18:00:00", "end": "2026-10-30T20:00:00"},
        {"start": "2026-10-31T10:00:00", "end": "2026-10-31T13:00:00"},
    ])


def read_course_notes() -> str:
    """Read simulated course notes describing the AI-agent assignment requirements."""
    return (
        "AAI 501: use LlamaIndex and Google's Gemini API. Include sensors "
        "that observe information and actuators that change the environment. "
        "Submit actual code; execution is not required."
    )


def append_record(filename: str, record: dict) -> str:
    """Persist a record to a local JSON Lines file."""
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    with (OUTPUT_DIR / filename).open("a", encoding="utf-8") as output:
        output.write(json.dumps(record) + "\n")
    return f"Saved to {OUTPUT_DIR / filename}"


# ACTUATORS: perform three distinct changes to local planning resources.
def create_task(title: str, due_date: str) -> str:
    """Create a local to-do task. due_date must use YYYY-MM-DD format."""
    datetime.strptime(due_date, "%Y-%m-%d")
    return append_record("tasks.jsonl", {"title": title, "due_date": due_date})


def reserve_study_block(topic: str, start: str, end: str) -> str:
    """Record a local study reservation in an available window; use ISO datetimes.

    This writes a local planner entry, not an event in an external calendar.
    """
    block_start, block_end = datetime.fromisoformat(start), datetime.fromisoformat(end)
    if block_end <= block_start:
        raise ValueError("The study block must end after it starts.")
    windows = json.loads(read_calendar())
    if not any(datetime.fromisoformat(w["start"]) <= block_start
               and block_end <= datetime.fromisoformat(w["end"]) for w in windows):
        raise ValueError("The study block must fit inside an available window.")
    return append_record("study_blocks.jsonl", {"topic": topic, "start": start, "end": end})


def save_study_plan(markdown: str) -> str:
    """Save the final study plan as a local Markdown document."""
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    plan_path = OUTPUT_DIR / "study_plan.md"
    plan_path.write_text(markdown, encoding="utf-8")
    return f"Study plan saved to {plan_path}"


async def main() -> None:
    load_dotenv(USD_ROOT / ".env")
    llm = GoogleGenAI(
        model="gemini-3.8-flash",
        api_key=os.environ["AAI_SECRET"],
    )
    tools = [FunctionTool.from_defaults(fn=fn) for fn in (
        read_assignments, read_calendar, read_course_notes,
        create_task, reserve_study_block, save_study_plan,
    )]
    agent = FunctionAgent(
        tools=tools,
        llm=llm,
        system_prompt=(
            "You are a study-planning agent working with a simulated dataset. "
            "Read assignments, calendar availability, and course notes first. "
            "Prioritize the earliest deadline. Use tools to create a task, "
            "reserve one available study block, and save a Markdown study plan. "
            "Only report actions as completed after the corresponding tool succeeds. "
            "Calendar reservations and tasks are local files. Explain any errors."
        ),
    )
    response = await agent.run(
        user_msg="Using the sample October 30–31 availability, plan work on my earliest assignment."
    )
    print(response)


# Optional notebook execution; leave commented for submission.
# await main()


ServerError: 503 Service Unavailable. {'message': '{\n  "error": {\n    "code": 503,\n    "message": "This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.",\n    "status": "UNAVAILABLE"\n  }\n}\n', 'status': 'Service Unavailable'}

### Question 2(a): Actual outcome and expected behavior

**Observed outcome:** An optional execution attempt returned `ServerError: 503 Service Unavailable`. Gemini reported that the model was experiencing high demand. No successful agent response, saved study plan, task, or study reservation was observed from that attempt. The recorded error is retained above. The assignment permits unexecuted code, so this service-availability issue does not change the implementation presented here.

**Expected behavior, not an observed result:** If the service is available and `await main()` is explicitly enabled, the agent should read the three input sources, prioritize the earliest assignment, create a task, reserve an available study block, and save a plan. Actual tool selection depends on the model. The API call is commented out in the submission to prevent accidental use of quota.

**Dependencies:** `llama-index-core`, `llama-index-llms-google-genai`, and `python-dotenv`. The code expects a local `.env` containing `AAI_SECRET`; the key itself is not included. Sensor inputs are sample data. Calendar and task actions write local files rather than external services.

- [LlamaIndex agents](https://developers.llamaindex.ai/python/framework/understanding/putting_it_all_together/agents/)
- [Google Gemini with LlamaIndex](https://ai.google.dev/gemini-api/docs/llama-index)
